# Estimating Park-and-Ride Lot Choice

This notebook estimates the single-segment park-and-ride lot choice model with Larch. The model is an interaction model, so it uses chooser-alternative expression values from the ActivitySim estimation data bundle.

Run ActivitySim in estimation mode first. The estimation run must include the park-and-ride fixes that preserve `pnr_zone_id` and configure it as the alternative ID.

## Load libraries

In [255]:
from pathlib import Path
from types import SimpleNamespace

import larch as lx
import pandas as pd

from activitysim.estimation.larch.general import (
    apply_coefficients,
    linear_utility_from_spec,
    update_coefficients,
)

lx.versions()

{'larch': '6.0.46',
 'sharrow': '2.16.2',
 'numpy': '2.2.6',
 'pandas': '2.3.2',
 'xarray': '2025.6.1',
 'numba': '0.62.0',
 'jax': 'not installed'}

## Locate the estimation data bundle

Change `EDB_DIRECTORY` if your ActivitySim output directory is elsewhere.

In [ ]:
modelname = "park_and_ride_lot_choice"
EDB_DIRECTORY = Path("output-est-mode/estimation_data_bundle/{modelname}/")
#EDB_DIRECTORY = Path(r"C:\workspace\asim_estimation_pnr\outputs_test2\estimation_data_bundle") / modelname


WindowsPath('C:/workspace/asim_estimation_pnr/outputs_test2/estimation_data_bundle/park_and_ride_lot_choice')

In [257]:
def read_edb_file(filename, **kwargs):
    path = EDB_DIRECTORY / filename
    parquet_path = path.with_suffix(".parquet")
    if parquet_path.exists():
        return pd.read_parquet(parquet_path, **kwargs)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path.name}. Re-run ActivitySim estimation after setting "
            f"the park-and-ride chooser ID, then coalesce the interaction EDB."
        )
    return pd.read_csv(path, **kwargs)

coefficient_candidates = [
    f"{modelname}_coefficients.csv",
    f"{modelname}_{modelname}_coeffs.csv",
    f"{modelname}_coeffs.csv",
]
coefficient_file = next(
    (filename for filename in coefficient_candidates if (EDB_DIRECTORY / filename).exists()),
    None,
)
if coefficient_file is None:
    raise FileNotFoundError(
        f"Could not find PNR coefficients in {EDB_DIRECTORY}. "
        f"Expected one of: {coefficient_candidates}"
    )

coefficients = read_edb_file(coefficient_file, index_col="coefficient_name")
spec = read_edb_file(f"{modelname}_SPEC.csv", comment="#")
alternatives = read_edb_file(f"{modelname}_alternatives_combined.csv")
choosers = read_edb_file(f"{modelname}_choosers_combined.csv")

if "pnr_zone_id" not in alternatives.columns:
    raise KeyError(f"Expected 'pnr_zone_id' in alternatives: {alternatives.columns.tolist()}")
chooser_id_column = choosers.columns[0]
if chooser_id_column not in alternatives.columns:
    raise ValueError(
        f"The alternatives EDB is missing chooser ID {chooser_id_column!r}. "
        "Re-run estimation after setting estimator.set_chooser_id(...) in the "
        "park-and-ride model."
    )

spec = spec.loc[spec.Expression.notna()].copy()
spec = spec.loc[~spec.Expression.str.startswith("_")].copy()
print(f"Using coefficient file: {coefficient_file}")
coefficients.head(), spec.head(), alternatives.head(), choosers.head()

Using coefficient file: park_and_ride_lot_choice_park_and_ride_lot_choice_coeffs.csv


(                         value constrain
 coefficient_name                        
 coef_zero             0.000000         T
 coef_unavailable   -999.000000         F
 coef_ivt_auto        -0.077806         F
 coef_ivt_transit     -0.048312         F
 coef_auto_op_costs    0.000000         F,
                    Label                               Description  \
 4        util_drive_time  round trip drive time from origin to lot   
 5        util_total_wait                           total wait time   
 6         util_walk_time                                 walk time   
 7              util_ivtt                   in vehcile transit time   
 8  util_log_lot_capacity                          log_lot_capacity   
 
                                        Expression        coefficient  
 4  @_d_pnr_time_skims_out + _d_pnr_time_skims_inb      coef_ivt_auto  
 5  @(ldt_skims['twtwpnr'] + dlt_skims['twtwpnr'])  coef_wait_transit  
 6  @(ldt_skims['auxwpnr'] + dlt_skims['auxwpnr'])  coef_walk

In [258]:
# Optional: exclude survey tours with a ~$17.90 toll on the home-to-lot leg (likely ferry); the toll is not modeled
# EXCLUDED_TOUR_IDS = [10431751, 9927843]
# alternatives = alternatives.loc[~alternatives[chooser_id_column].isin(EXCLUDED_TOUR_IDS)].copy()
# choosers = choosers.loc[~choosers[chooser_id_column].isin(EXCLUDED_TOUR_IDS)].copy()

## Build the Larch interaction model

The first two columns of `alternatives_combined` identify the chooser and alternative. The chooser table contains the observed choice written by ActivitySim. PNR uses `pnr_zone_id` as the alternative ID.

In [259]:
########Build the interaction model
alternative_id_column = "pnr_zone_id"

if alternative_id_column not in alternatives.columns:
    raise KeyError(f"Expected {alternative_id_column!r} in alternatives: {alternatives.columns.tolist()}")

key_columns = [chooser_id_column, alternative_id_column]
duplicate_count = int(alternatives.duplicated(key_columns).sum())
if duplicate_count:
    print(f"Removing {duplicate_count} duplicate chooser-alternative rows from the EDB")
    alternatives = alternatives.drop_duplicates(key_columns, keep="last").copy()

x_ca = alternatives.set_index(key_columns)
choosers = choosers.drop_duplicates(chooser_id_column, keep="last").copy()
x_co = choosers.loc[choosers[chooser_id_column].isin(alternatives[chooser_id_column])].set_index(chooser_id_column)

print(f"CA index: {x_ca.index.names}, unique={x_ca.index.is_unique}, rows={len(x_ca)}")
print(f"CO index: {x_co.index.name}, unique={x_co.index.is_unique}, rows={len(x_co)}")

choice_candidates = ["override_choice", "model_choice", "choice", "pnr_zone_id"]
choice_column = next((c for c in choice_candidates if c in x_co.columns), None)
if choice_column is None:
    raise KeyError(f"Could not find an observed choice column in {x_co.columns.tolist()}")

# Keep constant chooser-alternative variables two-dimensional for Larch.
d_ca = lx.Dataset.construct.from_idca(x_ca, crack=False)
d_co = lx.Dataset.construct.from_idco(x_co)
datatree = d_ca.merge(d_co)

model = lx.Model(datatree=datatree, compute_engine="numba")
alt_codes = datatree.dc.altids()
root_id = -1 if 0 in alt_codes else 0
model.initialize_graph(alternative_codes=alt_codes, root_id=root_id)

model.utility_ca = linear_utility_from_spec(
    spec,
    x_col="Label",
    p_col="coefficient",
    ignore_x=("#",),
    x_validator=set(x_ca.columns),
    expr_col="Expression",
)
apply_coefficients(coefficients, model)
model.choice_co_code = choice_column
model.availability_any = True
model.set_cap(50)

model.required_data()

CA index: ['tour_id', 'pnr_zone_id'], unique=True, rows=5733
CO index: tour_id, unique=True, rows=39


key,value
ca,"['util_drive_time', 'util_ivtt', 'util_log_lot_capacity', 'util_total_wait', 'util_walk_time']"
choice_co_code,'override_choice'
avail_any,True


## Review and diagnose the model

In [260]:
model, problems = model.doctor()
problems

<empty>


## Estimate

In [261]:
model.estimate(method="BHHH", options={"maxiter": 1000})

┣          loglike: np.float64(-91.7785059701134)
┣                x: coef_ivt_auto       -0.080334
┃                   coef_ivt_transit    -0.038466
┃                   coef_lot_capacity    0.493467
┃                   coef_wait_transit   -0.035961
┃                   coef_walk_transit   -0.049183
┃                   dtype: float64
┣        tolerance: np.float64(7.158925585950065e-06)
┣            steps: array([1. , 1. , 1. , 1. , 1. , 1. , 0.5, 1. , 1. , 0.5, 1. , 1. , 1. ,
┃                          0.5, 1. , 1. , 1. , 0.5])
┣          message: 'Optimization terminated successfully'
┣     elapsed_time: datetime.timedelta(microseconds=207510)
┣           method: 'BHHH'
┣          n_cases: 39
┣ iteration_number: 18

In [262]:
model.parameter_summary()
model.estimation_statistics()

<xmle.Elem 'div' with 1 children>

## Write estimation results

In [263]:
result_dir = EDB_DIRECTORY / "estimated"
data = SimpleNamespace(edb_directory=EDB_DIRECTORY, coefficients=coefficients)

update_coefficients(
    model,
    data,
    result_dir,
    output_file=f"{modelname}_coefficients.csv",
)

model.to_xlsx(
    result_dir / f"{modelname}_model_estimation.xlsx",
    data_statistics=False,
)

In [264]:
model.parameter_summary()

,Value,Std Err,t Stat,Signif,Null Value
Parameter,,,,,
coef_ivt_auto,-0.0803,0.0122,-6.59,***,0.00
coef_ivt_transit,-0.0385,0.00917,-4.20,***,0.00
coef_lot_capacity,0.493,0.179,2.76,**,0.00
coef_wait_transit,-0.0360,0.0173,-2.07,*,0.00
coef_walk_transit,-0.0492,0.0212,-2.32,*,0.00


## Next steps

Review the revised coefficient file, then copy `park_and_ride_lot_choice_coefficients.csv` into the active configuration directory before running ActivitySim in simulation mode.

In [265]:
pd.read_csv(result_dir / f"{modelname}_coefficients.csv")

,coefficient_name,value,constrain
0,coef_zero,0.000000,T
1,coef_unavailable,-999.000000,F
2,coef_ivt_auto,-0.080334,F
3,coef_ivt_transit,-0.038466,F
4,coef_auto_op_costs,0.000000,F
5,coef_walk_transit,-0.049183,F
6,coef_wait_transit,-0.035961,F
7,coef_xfer_transit,0.000000,F
8,coef_xwait_transit,0.000000,F
9,coef_small_lot,0.000000,F


In [266]:
# Compare mean predictor values for the chosen alternative vs. all alternatives
predictor_cols = [col for col in dict.fromkeys(spec["Label"]) if col in x_ca.columns]

chosen_by_chooser = x_co[choice_column].to_dict()
alt_ids = x_ca.index.get_level_values(alternative_id_column)
chooser_ids = x_ca.index.get_level_values(chooser_id_column)
is_chosen = alt_ids == chooser_ids.map(chosen_by_chooser)

comparison = pd.DataFrame(
    {
        "Chosen Alt Mean": x_ca.loc[is_chosen, predictor_cols].mean(),
        "All Alts Mean": x_ca[predictor_cols].mean(),
    }
)
comparison["Difference"] = comparison["Chosen Alt Mean"] - comparison["All Alts Mean"]
comparison.index.name = "Variable"
comparison

,Chosen Alt Mean,All Alts Mean,Difference
Variable,,,
util_drive_time,35.137123,112.517128,-77.380005
util_total_wait,32.398613,59.803665,-27.405052
util_walk_time,25.221519,41.669800,-16.448280
util_ivtt,56.971474,120.209511,-63.238037
util_log_lot_capacity,6.196738,5.187129,1.009609


In [267]:
x_ca.loc[is_chosen, predictor_cols]

,,util_drive_time,util_total_wait,util_walk_time,util_ivtt,util_log_lot_capacity
tour_id,pnr_zone_id,,,,,
9250500,66236,18.459089,50.630470,32.576805,111.647537,7.733684
4183761,66236,16.661467,12.919355,13.617260,116.280869,7.733684
4872110,66241,65.951553,11.834758,4.989394,4.994812,6.904751
3838500,66242,62.115025,44.710346,19.764475,91.464699,6.456770
11251300,66248,33.673740,26.785713,11.519560,39.772682,5.780744
1899200,66260,22.325586,22.000000,19.931280,39.917713,5.278115
6125890,66268,13.676851,18.823769,22.274820,85.118820,6.003887
14612890,66276,29.429062,14.970414,79.749962,9.989868,6.104793
3153800,66276,29.020229,49.262352,15.980379,75.591988,6.104793
